# Notebook 05 — RAFT (Retrieval-Augmented Fine-Tuning)

**Objectif** : entraîner un LoRA **RAFT** (pseudo-RAG au train : contextes FAISS), puis inférence alignée sur le test.

Sorties :
- `models/lora_adapter_raft/` (adaptateur LoRA pour la méthode RAFT)
- `results/raft_predictions.json` (approche RAFT)
- résumé métriques en fin de notebook

## 0. Vérification GPU

In [ ]:
# Vérification que le GPU est bien disponible avant de continuer
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Aucun GPU détecté !\n"
        "→ Allez dans Exécution > Modifier le type d'exécution > GPU, puis relancez."
    )

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU détecté  : {gpu_name}")
print(f"VRAM totale  : {gpu_mem_gb:.1f} Go")
print(f"CUDA version : {torch.version.cuda}")

## 1. Montage Google Drive

In [ ]:
# Montage du Drive et définition du chemin de base du projet
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 2. Installation des dépendances

In [ ]:
# Installation robuste Unsloth + Unsloth Zoo (versions cohérentes)
# IMPORTANT: exécuter cette cellule, puis redémarrer le runtime avant de continuer.
!pip uninstall -y unsloth unsloth_zoo trl transformers peft accelerate bitsandbytes xformers
!pip install --no-cache-dir -U git+https://github.com/unslothai/unsloth-zoo.git
!pip install --no-cache-dir -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print("Installation Unsloth terminée. Redémarre maintenant le runtime (Runtime > Restart runtime).")

In [ ]:
# Dépendances complémentaires RAFT (hors stack déjà gérée par Unsloth)
!pip install -q datasets trl sentence-transformers faiss-cpu rouge-score bert-score

## 3. Imports et configuration

In [ ]:
import os, json, time, re, string
import numpy as np
import torch
import faiss
from tqdm.notebook import tqdm
from datasets import Dataset
from sentence_transformers import SentenceTransformer

# Unsloth DOIT être importé avant trl/transformers/peft
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
MODELS_PATH    = os.path.join(BASE_PATH, 'models', 'lora_adapter_raft')
FAISS_PATH     = os.path.join(BASE_PATH, 'models', 'faiss_index')
RESULTS_PATH   = os.path.join(BASE_PATH, 'results')
for p in [MODELS_PATH, RESULTS_PATH]:
    os.makedirs(p, exist_ok=True)

BASE_MODEL   = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"
EMBED_MODEL  = "paraphrase-multilingual-MiniLM-L12-v2"
TOP_K        = 5
LORA_RANK    = 32
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
TARGET_MODS  = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
NUM_EPOCHS   = 5
LR           = 1e-4
SEED         = 42

# Profil auto selon GPU (priorite A100/H100)
gpu_name = torch.cuda.get_device_name(0).lower() if torch.cuda.is_available() else "cpu"
if "a100" in gpu_name or "h100" in gpu_name:
    MAX_SEQ_LEN = 1024
    BATCH_SIZE = 2
    GRAD_ACC_STEPS = 8
    MAX_CONTEXT_CHARS = 700
    GPU_PROFILE = "A100/H100"
elif "l4" in gpu_name:
    MAX_SEQ_LEN = 896
    BATCH_SIZE = 1
    GRAD_ACC_STEPS = 16
    MAX_CONTEXT_CHARS = 600
    GPU_PROFILE = "L4"
else:
    MAX_SEQ_LEN = 768
    BATCH_SIZE = 1
    GRAD_ACC_STEPS = 16
    MAX_CONTEXT_CHARS = 500
    GPU_PROFILE = "T4-safe"

EMBED_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print("Configuration RAFT prete.")
print(f"  GPU profile : {GPU_PROFILE} ({gpu_name})")
print(f"  max_seq_len : {MAX_SEQ_LEN} | batch/acc: {BATCH_SIZE}/{GRAD_ACC_STEPS}")
print(f"  Context max : {MAX_CONTEXT_CHARS}")
print(f"  Embed device: {EMBED_DEVICE}")
print(f"  LoRA output : {MODELS_PATH}")
print(f"  FAISS path  : {FAISS_PATH}")

## 4. Chargement des données

In [ ]:
def load_json(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  [OK] {path} ({len(data)} entrées)")
        return data
    except Exception as e:
        print(f"  [ERROR] {path}: {e}")
        return []

print("Chargement train/test + FAISS metadata...")
train_data = load_json(os.path.join(PROCESSED_PATH, 'train.json'))
test_data  = load_json(os.path.join(PROCESSED_PATH, 'test.json'))
corpus_meta = load_json(os.path.join(FAISS_PATH, 'metadata.json'))
index = faiss.read_index(os.path.join(FAISS_PATH, 'index.faiss'))
embed_model = SentenceTransformer(EMBED_MODEL, device=EMBED_DEVICE)
print(f"Train={len(train_data)} | Test={len(test_data)} | FAISS={index.ntotal}")

## 5. Formatage des exemples SFT (Alpaca ± contexte)

In [ ]:
# Construction pseudo-RAG pour l'entraînement (RAFT)
RAFT_TEMPLATE = (
    "### Instruction: Réponds à cette question en te basant uniquement sur le contexte fourni.\n"
    "### Context: {context}\n"
    "### Input: {question}\n"
    "### Response: {answer}"
)
NOISE_PATTERNS = [
    r"pour sauvegarder cet article.*$",
    r"connectez-vous.*$",
    r"abonnez-vous.*$",
    r"cookies?.*$",
]

def clean_text(txt):
    out = (txt or "").strip()
    for p in NOISE_PATTERNS:
        out = re.sub(p, "", out, flags=re.IGNORECASE | re.MULTILINE).strip()
    out = re.sub(r"\s+", " ", out)
    return out

def retrieve_context(question, k=TOP_K):
    q_emb = embed_model.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, idxs = index.search(q_emb, k)
    chunks = [corpus_meta[i] for i in idxs[0] if i < len(corpus_meta)]
    lines = []
    for c in chunks:
        txt = clean_text(c.get('text', c.get('context', '')))[:MAX_CONTEXT_CHARS]
        if len(txt) >= 30:
            lines.append(f"[{c.get('title','')[:60]}] {txt}")
    return "\n\n".join(lines) if lines else "Contexte indisponible."

def format_raft_example(item):
    q = item.get('question', '').strip()
    a = item.get('answer', '').strip()
    if not q or not a:
        return None
    ctx = retrieve_context(q, TOP_K)
    return {"text": RAFT_TEMPLATE.format(context=ctx, question=q, answer=a)}

formatted = []
for it in tqdm(train_data, desc="Pseudo-RAG train build"):
    row = format_raft_example(it)
    if row:
        formatted.append(row)

hf_dataset = Dataset.from_list(formatted)
split = hf_dataset.train_test_split(test_size=0.1, seed=SEED, shuffle=True)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"RAFT dataset: total={len(hf_dataset)} train={len(train_dataset)} val={len(eval_dataset)}")
print(train_dataset[0]["text"][:500])

## 6. Chargement du modèle et application de LoRA

In [ ]:
# Chargement de LLaMA 3.1 8B quantifié en 4-bit via Unsloth
print(f"Chargement de {BASE_MODEL}...")
print("(Téléchargement ~4 Go depuis HuggingFace, peut prendre 5-10 min)")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,          # Auto-détection (bfloat16 sur A100, float16 sur T4)
        load_in_4bit=True,   # Quantification 4-bit pour tenir en VRAM
    )
    print("Modèle de base chargé.")
except Exception as e:
    raise RuntimeError(f"Échec du chargement du modèle : {e}")

In [ ]:
# Application de l'adaptateur LoRA sur les couches cibles
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=TARGET_MODS,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Économie mémoire supplémentaire
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# Comptage des paramètres entraînables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Paramètres entraînables : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 7. Entraînement

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/tmp_checkpoints_raft",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    warmup_steps=30,
    max_grad_norm=1.0,
    learning_rate=LR,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)
print("Trainer RAFT prêt.")

In [ ]:
# Lancement de l'entraînement (environ 20-40 min sur T4 pour 300 exemples × 3 époques)
train_start = time.time()

try:
    trainer_stats = trainer.train()
    train_duration = time.time() - train_start
    print(f"\nEntraînement terminé en {train_duration/60:.1f} minutes")
    print(f"Loss finale      : {trainer_stats.training_loss:.4f}")
    print(f"Steps total      : {trainer_stats.global_step}")
except Exception as e:
    raise RuntimeError(f"Échec de l'entraînement : {e}")

## 8. Sauvegarde de l'adaptateur LoRA sur Drive

In [ ]:
# Sauvegarde de l'adaptateur LoRA (uniquement les poids delta, ~50 Mo)
try:
    model.save_pretrained(MODELS_PATH)
    tokenizer.save_pretrained(MODELS_PATH)
    print(f"Adaptateur LoRA sauvegardé : {MODELS_PATH}")

    # Listage des fichiers produits
    files = os.listdir(MODELS_PATH)
    for fname in files:
        fpath = os.path.join(MODELS_PATH, fname)
        size  = os.path.getsize(fpath) / 1024
        print(f"  {fname:<40} {size:>8.1f} Ko")
except Exception as e:
    print(f"[ERROR] Sauvegarde LoRA : {e}")

## 9. Inférence sur le test set

In [ ]:
# Inférence RAFT (retrieval au test, aligné avec l'entraînement)
FastLanguageModel.for_inference(model)
_STOP = ["\n### Instruction:", "\n### Input:", "\n### Response:", "\n### Context:"]

def _cleanup_answer(text):
    out = (text or "").strip()
    for m in _STOP:
        if m in out:
            out = out.split(m)[0].strip()
    if len(out) > 40 and out[-1] not in ".!?":
        cut = max(out.rfind('.'), out.rfind('!'), out.rfind('?'))
        if cut > 40:
            out = out[:cut+1]
    return out.strip()

def _conf(gen):
    if not getattr(gen, 'scores', None):
        return None
    probs = [torch.softmax(s[0], dim=-1).max().item() for s in gen.scores]
    return round(float(np.mean(probs)), 4) if probs else None

def generate_answer(question, max_new_tokens=320):
    ctx = retrieve_context(question, TOP_K)
    prompt = (
        "### Instruction: Réponds à cette question en te basant uniquement sur le contexte fourni.\n"
        f"### Context: {ctx}\n"
        f"### Input: {question}\n"
        "### Response:"
    )
    try:
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        start = time.time()
        with torch.no_grad():
            gen = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                min_new_tokens=20,
                do_sample=False,
                repetition_penalty=1.15,
                no_repeat_ngram_size=4,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
                return_dict_in_generate=True,
                output_scores=True,
            )
        latency_ms = round((time.time() - start) * 1000)
        prompt_len = inputs['input_ids'].shape[1]
        out_tokens = gen.sequences[0][prompt_len:]
        answer = tokenizer.decode(out_tokens, skip_special_tokens=True).strip()
        answer = _cleanup_answer(answer)
        return answer, latency_ms, _conf(gen), len(out_tokens) >= max_new_tokens
    except Exception as e:
        print(f"[ERROR] génération RAFT: {e}")
        return "", 0, None, False

print('Mode inférence RAFT prêt.')

In [ ]:
# Inférence RAFT sur tout le test set
finetuned_predictions = []

for item in tqdm(test_data, desc="Inférence RAFT"):
    question = item.get('question', '')
    true_answer = item.get('answer', '')
    predicted, latency, confidence, truncated = generate_answer(question)
    finetuned_predictions.append({
        "pair_id": item.get('pair_id', ''),
        "question": question,
        "predicted_answer": predicted,
        "true_answer": true_answer,
        "latency_ms": latency,
        "confidence": confidence,
        "truncated": truncated,
        "method": "raft"
    })

latencies = [p['latency_ms'] for p in finetuned_predictions if p['latency_ms'] > 0]
print(f"\nInférence RAFT terminée : {len(finetuned_predictions)} prédictions")
print(f"Latence moyenne         : {np.mean(latencies):.0f} ms" if latencies else "Latence : N/A")

In [ ]:
# Sauvegarde des prédictions RAFT
raft_path = os.path.join(RESULTS_PATH, 'raft_predictions.json')
try:
    with open(raft_path, 'w', encoding='utf-8') as f:
        json.dump(finetuned_predictions, f, ensure_ascii=False, indent=2)
    print(f"Prédictions RAFT sauvegardées : {raft_path}")
except Exception as e:
    print(f"[ERROR] Sauvegarde RAFT : {e}")

In [ ]:
# Mini-évaluation immédiate (même chemin que la cellule de sauvegarde RAFT)
raft_path = os.path.join(RESULTS_PATH, 'raft_predictions.json')
_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def _norm(t):
    t = (t or "").lower().strip()
    t = t.translate(str.maketrans('', '', string.punctuation))
    return " ".join(t.split())

def _f1(p, g):
    pt, gt = _norm(p).split(), _norm(g).split()
    if not pt or not gt:
        return 0.0
    inter = len(set(pt) & set(gt))
    if inter == 0:
        return 0.0
    pr, rc = inter / len(pt), inter / len(gt)
    return 2 * pr * rc / (pr + rc)

preds = [x.get("predicted_answer","") for x in finetuned_predictions]
refs  = [x.get("true_answer","") for x in finetuned_predictions]
em = np.mean([int(_norm(p) == _norm(g)) for p, g in zip(preds, refs)]) * 100
f1 = np.mean([_f1(p, g) for p, g in zip(preds, refs)]) * 100
rl = np.mean([_rouge.score(g if g else " ", p if p else " ")["rougeL"].fmeasure for p, g in zip(preds, refs)]) * 100
try:
    _, _, F = bert_score_fn(preds if preds else [" "], refs if refs else [" "], lang="fr",
                            model_type="distilbert-base-multilingual-cased", batch_size=32, verbose=False)
    bs = float(F.mean()) * 100
except Exception as e:
    print(f"[WARN] BERTScore indisponible: {e}")
    bs = 0.0
print("\n--- Mini-évaluation RAFT (NB05) ---")
print(f"EM={em:.1f}% | F1={f1:.1f}% | ROUGE-L={rl:.1f}% | BERTScore={bs:.1f}%")
if os.path.isfile(raft_path):
    print(f"Fichier JSON : {raft_path} ({os.path.getsize(raft_path)/1024:.1f} Ko)")
else:
    print(f"Fichier JSON (pas encore écrit) : {raft_path} — exécutez la cellule de sauvegarde ci-dessus.")

## 10. Résumé final RAFT

In [ ]:
# Résumé final RAFT
latencies = [p['latency_ms'] for p in finetuned_predictions if p['latency_ms'] > 0]
print("=" * 70)
print("RÉSUMÉ — Notebook 05 : RAFT")
print("=" * 70)
print(f"LoRA RAFT sauvegardé dans : {MODELS_PATH}")
print(f"Prédictions RAFT             : {os.path.join(RESULTS_PATH, 'raft_predictions.json')}")
print(f"Nombre prédictions test      : {len(finetuned_predictions)}")
print(f"Latence moyenne              : {np.mean(latencies):.0f} ms" if latencies else "Latence : N/A")
print("✔ Suite : 06_rag_rerank.ipynb → 07_function_calling.ipynb → 08_ft_plus_rag.ipynb → 09_evaluation.ipynb")
print("=" * 70)